# C2 · Qwen-Image-2.1 (Single-Stream DiT · block-causal · 편집 조건) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/C2_qwen_image21.ipynb)

원본: diffusers `QwenImage21Transformer2DModel` (`src/diffusers/models/transformers/transformer_qwenimage21.py`,
diffusers **v0.41.0** = commit `086bf95` 고정, 이후 main 의 변경은 tensor parallel 용이라 수치 동일).
체크포인트: HF `Qwen/Qwen-Image-2.1` (revision `d26bb61`), `transformer/config.json` 의 `_class_name` 이 위 클래스.
내 구현: `miniarch/models/qwen_image21.py`. VL 인코더(Qwen3-VL)는 구현하지 않고 랜덤 hidden state 를 넣습니다.

부품
- 한 줄 시퀀스: VL 시퀀스의 이미지 칸(1칸 = latent 2×2 = 토큰 4개)에 조건 이미지 latent 를, 맨 끝에 타깃 latent 를 끼움
- txt_in connector: zero-center RMSNorm(가중치+1) → Linear → GELU(tanh) → Linear
- 3축 RoPE: 텍스트는 세 축 같이 전진, 이미지 블록은 frame 고정 + 0 중심 h/w 격자 (음수 위치)
- block-causal attention: (key ≤ query) 또는 같은 이미지 블록. padding 텍스트는 key 에서만 제외. QK RMSNorm
- 공유 modulation 하나 → 블록마다 [scale, tanh(gate)] × (attn, MLP), shift 없음. SwiGLU MLP
- causal_condition: 텍스트·조건 이미지는 t=0 행으로 modulation → prefix 가 timestep·타깃과 무관 (KV cache 가능)

축소 config: 2층, 폭 64 (head 4 × 16), RoPE 축 (4, 6, 6), latent 8채널 → 출력 12채널, VL 폭 32.
VL 시퀀스 = 텍스트5 | 조건A 2칸(2×4) | 텍스트3 | 조건B 1칸(2×2) | 조건C 1칸(1×4, B 바로 옆) | 텍스트2 | 타깃 3칸(3×4) → 결합 토큰 38개.
- 조건 이미지 셋(편집 경로), 바로 붙은 두 블록(B·C 가 서로 못 보는지), 홀수 높이·높이 1 격자(음수 위치) 를 모두 지나게 했습니다.
- 두 번째 샘플은 텍스트 3자리를 padding 으로 막고, 첫 샘플은 조건 이미지 칸의 mask 를 꺼서 (무시되어야 함) 마스크 경로를 씁니다.
- 원본은 segment 마다 SDPA 를 부르고, 내 구현은 같은 마스크를 한 번에 만든 명시적 softmax 입니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. 학습 동작 확인 → 5. 단계 전용 자체 점검 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
if not os.environ.get('MINIARCH_SKIP_PIP'):   # 전체 실행 노트북이 한 번에 설치
    !pip -q install -U 'diffusers==0.41.0'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("C2")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('Qwen/Qwen-Image-2.1', '^(transformer_blocks\\.0\\.|img_in|txt_in|modulation|norm_out|proj_out|time_text_embed)', subfolder='transformer')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## 학습 동작 확인

forward 가 원본과 같아도 학습이 되는지는 따로 확인해야 합니다.
- **기울기 == 원본**: 같은 가중치·같은 손실로 원본과 내 구현을 backward 해서 파라미터 기울기를 비교합니다 (파라미터별 상대 오차 ≤ 1e-3).
  원본이 추론 전용 코드(no_grad)면 건너뜁니다.
- **학습 궤적 == 원본**: 같은 가중치·데이터·optimizer(SGD momentum 0.9, 30 step)로 원본과 내 구현을 나란히 학습해
  step 별 손실과 최종 가중치를 비교합니다 (상대차 ≤ 1e-3). Adam 은 아주 작은 잡음 차이를 키워서 궤적 비교에 쓰지 않습니다.
- **배치 하나 외우기**: 고정 배치 하나로 300 step 학습해 손실이 처음의 20% 이하로 내려가는지 봅니다 (LM은 다음 토큰 CE, 그 외는 고정 목표 MSE).
  NaN/inf 가 없어야 하고, 기울기를 한 번도 받지 못한 파라미터를 보여줍니다 (MoE 에서 안 뽑힌 expert 는 정상).

In [ ]:
import importlib, traceback
from miniarch.verify import training as TR
mod = importlib.import_module(S.CASE.mine_module)
V.set_deterministic(0)
ref, inputs = S.ref()
V.perturb_params(ref)
pair = mod.verify_pair(ref)
try:
    g = TR.grad_equivalence(ref, pair, inputs, dev)
    if g.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", g.note); report("기울기 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"비교한 파라미터 {g.compared}개, 최대 상대 오차 {g.max_rel:.2e} ({g.worst})", g.note)
        if g.skipped_ref:
            print("원본에서 기울기 없는 파라미터:", g.skipped_ref[:8])
        report("기울기 == 원본", g.ok, g.max_rel)
except Exception:
    traceback.print_exc(limit=4); report("기울기 == 원본", False)

In [ ]:
try:
    t = TR.train_parity(ref, pair, inputs, dev)
    if t.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", t.note); report("학습 궤적 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"{t.steps} step  손실 최대 상대차 {t.max_loss_rel:.2e}  최종 가중치 최대 상대차 {t.max_param_rel:.2e} ({t.worst_param})", t.note)
        print("  원본:", " ".join(f"{v:.4g}" for v in t.loss_ref[::6]))
        print("  내것:", " ".join(f"{v:.4g}" for v in t.loss_mine[::6]))
        report("학습 궤적 == 원본 (SGD 30 step)", t.ok, max(t.max_loss_rel, t.max_param_rel))
except Exception:
    traceback.print_exc(limit=4); report("학습 궤적 == 원본", False)

In [ ]:
try:
    o = TR.overfit_batch(pair["mine"], pair, inputs, dev)
    print(f"{o.kind}: {o.loss_start:.4f} → {o.loss_end:.4f}  (유한={o.finite})")
    print("  손실:", " ".join(f"{v:.3g}" for v in o.curve[::30]))
    if o.no_grad_params:
        print(f"  기울기를 못 받은 파라미터 {len(o.no_grad_params)}개:", o.no_grad_params[:8])
    report("배치 하나 외우기 (" + o.kind + ")", o.ok, o.loss_end / max(o.loss_start, 1e-12))
except Exception:
    traceback.print_exc(limit=4); report("배치 하나 외우기", False)

## 단계 전용 자체 점검

### attention map · 마스크 구조 · modulation 확인

- `attn_probs` [B, H, N, N]: 행 합 = 1, 막힌 칸(미래 텍스트, 뒤쪽 블록, padding key) 확률 = 0.
- 같은 이미지 블록 안은 전부 허용(양방향), 붙어 있는 조건 블록 B·C 는 B→C 방향이 막혀야 합니다.
- 텍스트→이미지 / 이미지→텍스트 / 이미지→이미지 확률 질량을 출력합니다.
- `causal_condition`: prefix 토큰의 scale/gate(t=0 행)는 두 샘플이 같고, 타깃 토큰은 샘플마다 달라야 합니다.

In [ ]:
from miniarch.models import qwen_image21 as Q
V.set_deterministic(0)
ref, inputs = S.ref()
m = Q.verify_pair(ref)["mine"]; V.copy_weights(ref, m); m = m.to(dev).eval()
x = {k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()}
names = [n for n, _ in m.named_modules() if n.endswith(("attn_probs", "_probe"))]
with V.capture(m, names) as st, torch.no_grad():
    y0 = m(**x)
allow, ids, tgt = st["mask_probe"], m.last_meta["image_ids"], m.last_meta["target_mask"]
N = ids.shape[0]
i_, j_ = torch.arange(N, device=dev)[:, None], torch.arange(N, device=dev)[None, :]
same = (ids[:, None] == ids[None, :]) & (ids[:, None] >= 0)
report("같은 이미지 블록 안은 전부 허용", bool(allow[:, same].all()), 0.0)
report("미래 칸(다른 블록) 은 전부 막힘", not bool(allow[:, (j_ > i_) & ~same].any()), 0.0)
b, c = (ids == 1).nonzero()[0, 0], (ids == 2).nonzero()[0, 0]
report("붙어 있는 조건 블록 B 는 C 를 못 봄", not bool(allow[:, b, c].any()), 0.0)
txt = ids < 0
for n in [n for n in names if n.endswith("attn_probs")]:
    p = st[n]
    err = (p.sum(-1) - 1).abs().max().item()
    leak = p.masked_fill(allow[:, None], 0).abs().max().item()
    t2i = p[:, :, txt][..., ~txt].sum(-1).mean().item()
    i2t = p[:, :, ~txt][..., txt].sum(-1).mean().item()
    i2i = p[:, :, ~txt][..., ~txt].sum(-1).mean().item()
    print(f"{n:<34} 텍스트→이미지 {t2i:.3f}  이미지→텍스트 {i2t:.3f}  이미지→이미지 {i2i:.3f}")
    report(f"{n} 행 합 = 1", err < 1e-5, err)
    report(f"{n} 막힌 칸 확률 = 0", leak == 0.0, leak)
for k in ("scale1", "gate1", "scale2", "gate2"):
    g = st[f"transformer_blocks.0.{k}_probe"]
    d_pre = (g[0, ~tgt] - g[1, ~tgt]).abs().max().item()
    d_tgt = (g[0, tgt] - g[1, tgt]).abs().max().item()
    print(f"{k}: prefix |평균|={g[:, ~tgt].abs().mean():.3f}  타깃 |평균|={g[:, tgt].abs().mean():.3f}")
    report(f"{k} prefix 행은 샘플 간 동일 (t=0)", d_pre == 0.0, d_pre)
    report(f"{k} 타깃 행은 샘플마다 다름 (실제 t)", d_tgt > 1e-4, d_tgt)

### prefix 불변성 (block-causal + causal_condition → KV cache 근거)

타깃 latent 와 timestep 을 바꿔도 텍스트·조건 이미지 위치 출력은 그대로여야 하고, 타깃 출력은 바뀌어야 합니다.
ablation: `causal_condition=False` 면 timestep 만 바꿔도 prefix 가 바뀌고, `block_causal=False` 면 타깃만 바꿔도 prefix 가 바뀝니다.

In [ ]:
import dataclasses, math
nt = math.prod(x["img_shapes"][0][-1])
hs = x["hidden_states"].clone(); hs[:, -nt:] = torch.randn_like(hs[:, -nt:])
x_tgt = {**x, "hidden_states": hs}
x_t = {**x, "timestep": 1.0 - x["timestep"]}   # [0.3, 0.8] → [0.7, 0.2] (두 샘플 모두 바뀜)
with torch.no_grad():
    y1 = m(**{**x_tgt, "timestep": x_t["timestep"]})
d_pre, d_tgt = (y1 - y0)[:, ~tgt].abs().max().item(), (y1 - y0)[:, tgt].abs().max().item()
report("타깃·t 바꿔도 prefix 출력 불변", d_pre < 1e-5, d_pre)
report("타깃 출력은 바뀜", d_tgt > 1e-3, d_tgt)
for sw, xx in (("causal_condition", x_t), ("block_causal", x_tgt)):
    m2 = Q.QwenImage21(dataclasses.replace(m.config, **{sw: False})).to(dev).eval()
    m2.load_state_dict(m.state_dict())
    with torch.no_grad():
        d = (m2(**xx) - m2(**x))[:, ~tgt].abs().max().item()
    report(f"{sw}=False 면 prefix 가 바뀜", d > 1e-4, d)

### padding · 무시되는 입력 · CFG 경로

- padding 텍스트 위치의 VL hidden 을 바꿔도 유효 토큰 출력은 그대로 (key 에서 빠짐, 자기 행만 바뀜).
- 조건 이미지 칸의 VL hidden 은 latent 로 덮어써지므로 바꿔도 출력이 전혀 안 바뀜.
- CFG: 빈 조건(VL hidden 0) 출력과 섞은 `uncond + 4·(cond − uncond)` 가 유한하고 cond 와 다른지 확인.

In [ ]:
em, im = x["encoder_hidden_states_mask"], x["img_mask"][:, :x["encoder_hidden_states_mask"].shape[1]]
e = x["encoder_hidden_states"]
e_pad = e.clone(); pad = (~em) & (~im); e_pad[pad] = torch.randn_like(e_pad[pad]) * 3
e_slot = e.clone(); e_slot[im] = torch.randn_like(e_slot[im]) * 3
with torch.no_grad():
    y_pad = m(**{**x, "encoder_hidden_states": e_pad})
    y_slot = m(**{**x, "encoder_hidden_states": e_slot})
    y_null = m(**{**x, "encoder_hidden_states": torch.zeros_like(e)})
kv = m.last_meta["key_valid"]
d = (y_pad - y0)[kv].abs().max().item()
report("padding 텍스트 내용은 유효 토큰에 영향 없음", d < 1e-5, d)
d = (y_pad - y0)[~kv].abs().max().item()
report("padding 토큰 자기 출력은 바뀜", d > 1e-4, d)
d = (y_slot - y0).abs().max().item()
report("조건 이미지 칸의 VL hidden 은 무시됨", d < 1e-6, d)
cfg = y_null + 4.0 * (y0 - y_null)
d = (cfg - y0)[:, tgt].abs().max().item()
report("CFG 출력 유한 + cond 와 다름", bool(torch.isfinite(cfg).all()) and d > 1e-3, d)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/C2_qwen_image21.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_C2_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")